In [1]:
"""
Make three collages from the four interpretability figures in this folder,
stacked vertically, three CNNs first and ViT last:
    1. ConvNeXtBase_gradcam
    2. EfficientNetV2M_gradcam
    3. InceptionResNetV2_gradcam
    4. ViT-Base_attention
Outputs (written into this folder):
    <foldername>_collage.png   1200 dpi raster  (submission)
    <foldername>_collage.tif   600 dpi LZW TIFF (submission)
    <foldername>_collage.svg   vector           (Word draft)
The PNG and TIFF collages stack the same-format panels with Pillow at full
resolution, no resampling. The SVG collage stacks the four SVG panels with
svgutils so the text and lines stay vector. Drop this script into each
*_interpretability folder and run it (python make_collage.py).

Requires: pip install svgutils
Keep PNG_DPI / TIFF_DPI equal to the panel export dpi so the dpi metadata
written here matches the actual pixel resolution of the panels.
"""
import os
import re
from PIL import Image
import svgutils.transform as sg

# Pillow's decompression-bomb cap is lifted for the large raster panels.
Image.MAX_IMAGE_PIXELS = None

GAP_PX = 0                   # vertical gap between stacked panels
BG_COLOR = (255, 255, 255)   # white, matches the figure background
PNG_DPI = 1200               # must match EXPORT_DPI['png'] in the panel notebooks
TIFF_DPI = 600               # must match EXPORT_DPI['tiff'] in the panel notebooks

# Stacking order: three CNNs, then ViT. Basenames only; extension added per format.
ORDERED_BASENAMES = [
    "ConvNeXtBase_gradcam",
    "EfficientNetV2M_gradcam",
    "InceptionResNetV2_gradcam",
    "ViT-Base_attention",
]


def _flatten_to_rgb(im):
    """Composite any transparency onto white and return an RGB image."""
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        im = im.convert("RGBA")
        bg = Image.new("RGB", im.size, BG_COLOR)
        bg.paste(im, mask=im.split()[-1])
        return bg
    return im.convert("RGB")


def _stack_raster(here, ext, dpi, out_path):
    """Stack same-format raster panels vertically and save with dpi metadata."""
    images = []
    for base in ORDERED_BASENAMES:
        path = os.path.join(here, f"{base}.{ext}")
        if not os.path.exists(path):
            print(f"  WARNING: {base}.{ext} not found, skipping.")
            continue
        im = Image.open(path)
        images.append(_flatten_to_rgb(im))
        print(f"  loaded {base}.{ext}  ({im.width} x {im.height})")
    if not images:
        raise FileNotFoundError(f"No .{ext} panels found in {here}")

    target_w = max(im.width for im in images)
    total_h = sum(im.height for im in images) + GAP_PX * (len(images) - 1)
    canvas = Image.new("RGB", (target_w, total_h), BG_COLOR)
    y = 0
    for im in images:
        x = (target_w - im.width) // 2          # centre narrower panels
        canvas.paste(im, (x, y))
        y += im.height + GAP_PX

    save_kwargs = {"dpi": (dpi, dpi)}
    if ext in ("tif", "tiff"):
        save_kwargs["compression"] = "tiff_lzw"
    canvas.save(out_path, **save_kwargs)
    print(f"  Saved {out_path}  ({canvas.width} x {canvas.height}, {dpi} dpi)")


def _parse_pt(value):
    """Strip units from an svgutils width/height string and return a float."""
    return float(re.sub(r"[^0-9.]+", "", str(value)))


def _stack_svg(here, out_path):
    """Stack the four SVG panels vertically into one vector SVG."""
    figs, widths, heights = [], [], []
    for base in ORDERED_BASENAMES:
        path = os.path.join(here, f"{base}.svg")
        if not os.path.exists(path):
            print(f"  WARNING: {base}.svg not found, skipping.")
            continue
        f = sg.fromfile(path)
        figs.append(f)
        widths.append(_parse_pt(f.width))
        heights.append(_parse_pt(f.height))
        print(f"  loaded {base}.svg  ({widths[-1]:.0f} x {heights[-1]:.0f} pt)")
    if not figs:
        raise FileNotFoundError(f"No .svg panels found in {here}")

    total_w = max(widths)
    total_h = sum(heights)
    root = sg.SVGFigure()
    root.set_size((f"{total_w}pt", f"{total_h}pt"))
    root.root.set("viewBox", f"0 0 {total_w} {total_h}")

    plots, y = [], 0.0
    for f, w, h in zip(figs, widths, heights):
        element = f.getroot()
        element.moveto((total_w - w) / 2.0, y)   # centre narrower panels
        plots.append(element)
        y += h
    root.append(plots)
    root.save(out_path)
    print(f"  Saved {out_path}  ({total_w:.0f} x {total_h:.0f} pt, vector)")


def main():
    here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    folder_name = os.path.basename(here.rstrip("/\\")) or "interpretability"

    print("PNG collage:")
    _stack_raster(here, "png", PNG_DPI, os.path.join(here, f"{folder_name}_collage.png"))
    print("\nTIFF collage:")
    _stack_raster(here, "tif", TIFF_DPI, os.path.join(here, f"{folder_name}_collage.tif"))
    print("\nSVG collage:")
    _stack_svg(here, os.path.join(here, f"{folder_name}_collage.svg"))


if __name__ == "__main__":
    main()

PNG collage:
  loaded ConvNeXtBase_gradcam.png  (59387 x 12300)
  loaded EfficientNetV2M_gradcam.png  (59387 x 12300)
  loaded InceptionResNetV2_gradcam.png  (59387 x 12300)
  loaded ViT-Base_attention.png  (59387 x 12300)
  Saved C:\Users\hrushisanap\OneDrive\Desktop\papers\revision\interpretability_new\dermamnist_interpretability\dermamnist_interpretability_collage.png  (59387 x 49200, 1200 dpi)

TIFF collage:
  loaded ConvNeXtBase_gradcam.tif  (29693 x 6150)
  loaded EfficientNetV2M_gradcam.tif  (29693 x 6150)
  loaded InceptionResNetV2_gradcam.tif  (29693 x 6150)
  loaded ViT-Base_attention.tif  (29693 x 6150)
  Saved C:\Users\hrushisanap\OneDrive\Desktop\papers\revision\interpretability_new\dermamnist_interpretability\dermamnist_interpretability_collage.tif  (29693 x 24600, 600 dpi)

SVG collage:
  loaded ConvNeXtBase_gradcam.svg  (3563 x 738 pt)
  loaded EfficientNetV2M_gradcam.svg  (3563 x 738 pt)
  loaded InceptionResNetV2_gradcam.svg  (3563 x 738 pt)
  loaded ViT-Base_attentio